# Topological-stability diagnostics for caged states

This notebook implements the first three numerical tests proposed for the topological-stability program:

1. **one-parameter branch tracking**, separating a continued cage eigenstate from the unchanged pre-quench state;
2. **random multi-parameter ensembles**, comparing compatible and incompatible local directions;
3. **the linearized obstruction map**, including both the boundary-cancellation equation and the internal eigenvalue equation.

The notebook begins with a transparent four-state toy cancellation network, then applies the same workflow to the known square-QDM \(4\times4\), \(W=(0,0)\), \((\kappa,Z)=(0,4)\) cage.

The tests establish exact or structural stability inside a specified perturbation class. They do **not** yet establish a topological invariant; that is the next stage of the project.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the notebook runnable both from the repository root and from this folder.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageSearchConfig,
    cage_compatibility_hierarchy_from_hamiltonians,
    CageSearcher,
    combine_perturbations_from_coefficients,
    diagnose_cage_stability,
    estimate_power_law_exponent,
    linearized_cage_obstruction_from_hamiltonians,
    random_cage_stability_ensemble,
    scan_cage_stability_branch,
    scan_support_eigenstate_branch,
)
from qlinks.models import SquareQDMModel

## 1. Toy cancellation network

The support contains two Fock-space vertices.  The boundary map

\[
B_0=\begin{pmatrix}1&1\\0&0\end{pmatrix}
\]

annihilates the antisymmetric cage vector \((1,-1)/\sqrt2\).  We compare:

- a **strongly compatible** perturbation that keeps the same vector;
- a **structure-compatible** perturbation that rotates the boundary kernel and therefore deforms the cage vector;
- an **incompatible** perturbation that adds an independent boundary row and removes the kernel.


In [ ]:
def assemble_hamiltonian(boundary, internal=None, external=None):
    if internal is None:
        internal = np.zeros((2, 2), dtype=np.complex128)
    if external is None:
        external = np.diag([2.0, 3.0]).astype(np.complex128)
    return np.block([[internal, boundary.conj().T], [boundary, external]])

base_boundary = np.array([[1.0, 1.0], [0.0, 0.0]], dtype=np.complex128)
base_hamiltonian = assemble_hamiltonian(base_boundary)

strong_perturbation = assemble_hamiltonian(
    np.array([[1.0, 1.0], [0.0, 0.0]], dtype=np.complex128),
    internal=np.eye(2, dtype=np.complex128),
    external=np.zeros((2, 2), dtype=np.complex128),
)
structure_perturbation = assemble_hamiltonian(
    np.array([[0.0, 1.0], [0.0, 0.0]], dtype=np.complex128),
    external=np.zeros((2, 2), dtype=np.complex128),
)
incompatible_perturbation = assemble_hamiltonian(
    np.array([[0.0, 0.0], [1.0, 0.0]], dtype=np.complex128),
    external=np.zeros((2, 2), dtype=np.complex128),
)

support = (0, 1)
cage_state = np.array([1.0, -1.0], dtype=np.complex128) / np.sqrt(2.0)

baseline = diagnose_cage_stability(
    base_hamiltonian,
    support,
    state=cage_state,
    tolerance=1.0e-12,
)
baseline.to_summary_dict()

In [ ]:
parameters = np.linspace(0.0, 1.0, 21)
structure_branch = scan_cage_stability_branch(
    base_hamiltonian,
    structure_perturbation,
    support,
    parameters,
    reference_state=cage_state,
    tolerance=1.0e-12,
)
incompatible_branch = scan_cage_stability_branch(
    base_hamiltonian,
    incompatible_perturbation,
    support,
    parameters,
    reference_state=cage_state,
    tolerance=1.0e-12,
)

branch_table = pd.DataFrame(
    {
        "lambda": parameters,
        "structure_dimension": structure_branch.invariant_dimensions,
        "structure_fixed_residual": structure_branch.fixed_state_full_residuals,
        "structure_continued_residual": structure_branch.continued_full_residuals,
        "incompatible_dimension": incompatible_branch.invariant_dimensions,
    }
)
branch_table.head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    structure_branch.parameters,
    structure_branch.fixed_state_full_residuals,
    marker="o",
    label="unchanged pre-quench state",
)
plt.plot(
    structure_branch.parameters,
    structure_branch.continued_full_residuals,
    marker="s",
    label="continued cage eigenstate",
)
plt.yscale("log")
plt.xlabel(r"deformation $\lambda$")
plt.ylabel("full eigenstate residual")
plt.title("A cage branch can survive while the original state stops being stationary")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
toy_obstruction = linearized_cage_obstruction_from_hamiltonians(
    base_hamiltonian,
    (strong_perturbation, structure_perturbation, incompatible_perturbation),
    support,
    cage_state,
    coefficient_field="real",
    tolerance=1.0e-12,
)

pd.DataFrame(
    [item.to_summary_dict() for item in toy_obstruction.perturbation_diagnostics]
)

In [ ]:
toy_compatible = random_cage_stability_ensemble(
    base_hamiltonian,
    (strong_perturbation, structure_perturbation),
    support,
    strengths=(0.1, 0.5, 1.0),
    n_samples=32,
    reference_state=cage_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=7,
    tolerance=1.0e-12,
)
toy_incompatible = random_cage_stability_ensemble(
    base_hamiltonian,
    (incompatible_perturbation,),
    support,
    strengths=(0.1, 0.5, 1.0),
    n_samples=32,
    reference_state=cage_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=7,
    tolerance=1.0e-12,
)

pd.DataFrame(
    [
        {"ensemble": "compatible", **item.to_summary_dict()}
        for item in toy_compatible.aggregates
    ]
    + [
        {"ensemble": "incompatible", **item.to_summary_dict()}
        for item in toy_incompatible.aggregates
    ]
)

## 2. Square QDM \(4\times4\), \(W=(0,0)\)

We now use the exact full-basis cage search.  The chosen record belongs to the known \((0,4)\) family.  Its candidate support contains the complete nine-dimensional caged invariant subspace, so the static report distinguishes:

- the boundary-kernel dimension;
- the internally invariant cage dimension;
- the selected record's eigenstate residual and weight inside that invariant subspace.


In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(search_type="type1", tolerance=1.0e-10),
).run()

square_record = square_search[(0, 4), 0]
square_baseline = diagnose_cage_stability(
    square_build.hamiltonian,
    square_record.support,
    state=square_record.local_state,
    tolerance=1.0e-10,
)

{
    "hilbert_dimension": square_build.hamiltonian.shape[0],
    "counts_by_signature": square_search.counts_by_signature,
    **square_baseline.to_summary_dict(),
}

### Local perturbation basis and tangent-space codimension

The raw parameter basis consists of all individually built kinetic and potential plaquette operators.  The full tangent obstruction enforces

$$
B\,\delta\phi+\delta B\,\phi=0,
\qquad
(A-E)\delta\phi+(\delta A-\delta E)\phi=0,
\qquad
\langle\phi|\delta\phi\rangle=0.
$$

Its nullspace gives the real coefficient combinations that preserve a caged eigenstate to first order.  This is stronger than testing only $P_{\ker B^\dagger}\delta B|\phi\rangle=0$.


In [ ]:
local_operators = square_build.kinetic_operators + square_build.potential_operators
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
local_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in local_operators
)

square_obstruction = linearized_cage_obstruction_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    square_record.support,
    square_record.local_state,
    coefficient_field="real",
    tolerance=1.0e-10,
)

{
    "n_local_parameters": square_obstruction.n_parameters,
    "obstruction_rank": square_obstruction.rank,
    "compatible_tangent_dimension": square_obstruction.compatible_dimension,
    "compatible_tangent_codimension": (
        square_obstruction.n_parameters - square_obstruction.compatible_dimension
    ),
    "individually_boundary_compatible_terms": sum(
        item.first_order_boundary_compatible
        for item in square_obstruction.perturbation_diagnostics
    ),
    "individually_full_tangent_compatible_terms": sum(
        item.first_order_eigenstate_compatible
        for item in square_obstruction.perturbation_diagnostics
    ),
    "individually_state_preserving_terms": sum(
        item.preserves_state
        for item in square_obstruction.perturbation_diagnostics
    ),
}

The important object is the **coefficient-space nullspace**, not the list of individually compatible terms.  In this benchmark, compatible deformations arise through collective combinations of local operators.


In [ ]:
compatible_term_matrices = combine_perturbations_from_coefficients(
    local_term_matrices,
    square_obstruction.compatible_coefficient_basis,
)
len(compatible_term_matrices)

### Exact finite-amplitude ensemble test

We compare two ensembles:

- random directions inside the full first-order compatible coefficient subspace;
- random directions in the unrestricted raw local-term basis.

The survival test requires an exact caged invariant state and a matched cage eigenstate with overlap at least \(0.5\) with the selected reference record.  The threshold is a continuation diagnostic, not a claimed universal constant.


In [ ]:
compatible_strengths = (1.0e-4, 1.0e-2, 1.0, 10.0)
control_strengths = (1.0e-6, 1.0e-4, 1.0e-2)

square_compatible_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    compatible_term_matrices,
    square_record.support,
    strengths=compatible_strengths,
    n_samples=3,
    reference_state=square_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=11,
    tolerance=1.0e-9,
)
square_control_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    local_term_matrices,
    square_record.support,
    strengths=control_strengths,
    n_samples=3,
    reference_state=square_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=11,
    tolerance=1.0e-9,
)

square_ensemble_table = pd.DataFrame(
    [
        {"ensemble": "tangent-compatible", **item.to_summary_dict()}
        for item in square_compatible_ensemble.aggregates
    ]
    + [
        {"ensemble": "unrestricted local control", **item.to_summary_dict()}
        for item in square_control_ensemble.aggregates
    ]
)
square_ensemble_table

In [ ]:
plt.figure(figsize=(7, 4))
for ensemble_name, group in square_ensemble_table.groupby("ensemble"):
    plt.plot(
        group["strength"],
        group["survival_fraction"],
        marker="o",
        label=ensemble_name,
    )
plt.xscale("log")
plt.ylim(-0.05, 1.05)
plt.xlabel("perturbation strength")
plt.ylabel("exact cage survival fraction")
plt.title("Compatible coefficient directions versus generic local directions")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
for ensemble_name, group in square_ensemble_table.groupby("ensemble"):
    plt.plot(
        group["strength"],
        group["minimum_interference_gap"],
        marker="o",
        label=ensemble_name,
    )
plt.xscale("log")
plt.yscale("log")
plt.xlabel("perturbation strength")
plt.ylabel(r"minimum nonzero singular value $\Delta_{\mathrm{I}}$")
plt.title("The interference singular gap detects rank-changing controls")
plt.legend()
plt.tight_layout()
plt.show()

## 3. What the first results mean

For the current archive and the selected square-QDM record, the executed notebook finds:

- a nine-dimensional exact caged invariant subspace on the record's candidate support;
- 48 real local-term parameters;
- an 11-dimensional full first-order compatible coefficient subspace, hence codimension 37;
- no single raw local term that by itself satisfies the full selected-eigenstate tangent condition;
- exact finite-amplitude survival for the sampled collective compatible directions;
- immediate destruction of the exact cage for generic random local directions, even at very small strength.

This is strong evidence for a **collectively defined structurally stable family**.  It is not yet evidence for topology, because we have not shown a discrete invariant or a no-go obstruction to deforming the family into a trivial localized state.

The next numerical stage should therefore compare the suspected robust and fragile cage records using the same tangent codimension, interference-gap statistics, and explicit compatible-path searches.  Only after that comparison should we implement candidate chiral-index and Fock-space homology diagnostics.


## 4. Robust versus fragile records: compatibility hierarchy

The first-order obstruction map is only the tangent equation at the undeformed Hamiltonian.  It can overestimate the set of finite-amplitude cage-preserving paths, especially when the support Hamiltonian has a degenerate eigenspace.

We therefore compare two nested coefficient spaces:

1. the **formal first-order continuation space**, which permits a correction to the cage vector;
2. the **exact fixed-state space**, for which the same compact vector remains an eigenstate of every affine Hamiltonian $H_0+\lambda V$.

The quotient between them consists of tangent-only directions whose finite-amplitude integrability must be tested explicitly.


In [ ]:
robust_record = square_search[(0, 4), 0]
fragile_record = square_search[(0, 6), 0]

robust_hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    robust_record.support,
    robust_record.local_state,
    coefficient_field="real",
    tolerance=1.0e-10,
)
fragile_hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    fragile_record.support,
    fragile_record.local_state,
    coefficient_field="real",
    tolerance=1.0e-10,
)

pd.DataFrame(
    [
        {"record": "robust candidate (0, 4)", **robust_hierarchy.to_summary_dict()},
        {"record": "fragile candidate (0, 6)", **fragile_hierarchy.to_summary_dict()},
    ]
)

The two records have the same 11-dimensional formal tangent space, so tangent codimension alone does not distinguish them.  The finite hierarchy does:

- for the $(0,4)$ cage, all 11 tangent directions already preserve the selected state exactly;
- for the $(0,6)$ cage, only 7 directions are exact, leaving 4 tangent-only directions.

This gives a concrete numerical meaning to “robust” and “fragile” before introducing a topological invariant.


In [ ]:
robust_fixed_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    robust_hierarchy.fixed_state.compatible_coefficient_basis,
)
fragile_fixed_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    fragile_hierarchy.fixed_state.compatible_coefficient_basis,
)
fragile_tangent_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    fragile_hierarchy.first_order.compatible_coefficient_basis,
)
fragile_tangent_only_directions = combine_perturbations_from_coefficients(
    local_term_matrices,
    fragile_hierarchy.tangent_only_coefficient_basis,
)

{
    "robust_exact_directions": len(robust_fixed_directions),
    "fragile_exact_directions": len(fragile_fixed_directions),
    "fragile_formal_tangent_directions": len(fragile_tangent_directions),
    "fragile_tangent_only_directions": len(fragile_tangent_only_directions),
}

### Finite-amplitude integrability of a tangent-only direction

For a direction that satisfies the formal first-order equation but is not exactly state-preserving, we continue the closest eigenspace of the internal support Hamiltonian.  Inside each degenerate eigenspace, the algorithm chooses the state with minimum boundary leakage.  This avoids mistaking an arbitrary basis choice inside a degenerate level for the physical near-cage branch.


In [ ]:
path_parameters = np.concatenate(([0.0], np.logspace(-5, -2, 7)))

robust_fixed_branch = scan_support_eigenstate_branch(
    square_build.hamiltonian,
    robust_fixed_directions[0],
    robust_record.support,
    path_parameters,
    reference_state=robust_record.local_state,
    tolerance=1.0e-11,
)
fragile_fixed_branch = scan_support_eigenstate_branch(
    square_build.hamiltonian,
    fragile_fixed_directions[0],
    fragile_record.support,
    path_parameters,
    reference_state=fragile_record.local_state,
    tolerance=1.0e-11,
)
fragile_tangent_only_branch = scan_support_eigenstate_branch(
    square_build.hamiltonian,
    fragile_tangent_only_directions[0],
    fragile_record.support,
    path_parameters,
    reference_state=fragile_record.local_state,
    tolerance=1.0e-11,
)

fragile_leakage_exponent = estimate_power_law_exponent(
    fragile_tangent_only_branch.parameters,
    fragile_tangent_only_branch.boundary_residuals,
    minimum_parameter=1.0e-6,
    minimum_residual=1.0e-13,
)

pd.DataFrame(
    {
        "lambda": path_parameters,
        "robust_exact_residual": robust_fixed_branch.boundary_residuals,
        "fragile_exact_residual": fragile_fixed_branch.boundary_residuals,
        "fragile_tangent_only_residual": (
            fragile_tangent_only_branch.boundary_residuals
        ),
        "fragile_tangent_only_cage_dimension": [
            point.invariant_cage_dimension
            for point in fragile_tangent_only_branch.points
        ],
    }
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    path_parameters[1:],
    np.maximum(robust_fixed_branch.boundary_residuals[1:], 1.0e-16),
    marker="o",
    label="(0, 4): exact compatible direction",
)
plt.plot(
    path_parameters[1:],
    np.maximum(fragile_fixed_branch.boundary_residuals[1:], 1.0e-16),
    marker="s",
    label="(0, 6): exact compatible direction",
)
plt.plot(
    path_parameters[1:],
    fragile_tangent_only_branch.boundary_residuals[1:],
    marker="^",
    label="(0, 6): tangent-only direction",
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"deformation $|\lambda|$")
plt.ylabel(r"minimum support-eigenstate leakage $\|B\phi\|$")
plt.title("A formal tangent direction develops finite-amplitude leakage")
plt.legend()
plt.tight_layout()
plt.show()

{
    "fragile_tangent_only_leakage_exponent": fragile_leakage_exponent,
    "nonzero_path_cage_dimensions": tuple(
        point.invariant_cage_dimension
        for point in fragile_tangent_only_branch.points[1:]
    ),
}

The tangent-only leakage scales as approximately $\lambda^2$, while the exact directions remain at numerical precision.  Thus the fragile record satisfies the linearized solvability condition but immediately loses its exact cage at every nonzero deformation on this path.  This is precisely the failure that a quench of only hand-selected fixed-state-compatible terms cannot reveal.


In [ ]:
fragile_tangent_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    fragile_tangent_directions,
    fragile_record.support,
    strengths=(1.0e-2, 1.0),
    n_samples=3,
    reference_state=fragile_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=23,
    tolerance=1.0e-9,
)
fragile_fixed_ensemble = random_cage_stability_ensemble(
    square_build.hamiltonian,
    fragile_fixed_directions,
    fragile_record.support,
    strengths=(1.0e-2, 1.0),
    n_samples=3,
    reference_state=fragile_record.local_state,
    target_dimension=1,
    minimum_subspace_overlap=0.5,
    random_seed=23,
    tolerance=1.0e-9,
)

pd.DataFrame(
    [
        {"ensemble": "formal tangent", **item.to_summary_dict()}
        for item in fragile_tangent_ensemble.aggregates
    ]
    + [
        {"ensemble": "exact fixed-state", **item.to_summary_dict()}
        for item in fragile_fixed_ensemble.aggregates
    ]
)

## 5. Current conclusion and next invariant test

The comparison now supplies a reproducible finite-size distinction:

- **robust $(0,4)$ record:** formal tangent space = exact affine compatibility space;
- **fragile $(0,6)$ record:** formal tangent space strictly contains the exact space, and the extra directions acquire quadratic leakage.

This still does not prove topology.  The next diagnostic should explain *why* the $(0,4)$ compatibility equations close exactly.  The most targeted next step is to construct the support–boundary interference network, compute its structured row dependencies and cycle/holonomy data, and test which of those quantities remains unchanged across the 11-dimensional exact family but fails for the four non-integrable directions.


## 6. IPR-resolved records, regional kernels, and the relative quotient

The same stability objective is continued here rather than in separate notebooks.  We first rotate the ninefold $(0,4)$ eigenspace to a deterministic IPR-localized basis.  The first eight compact records define eight locality-restricted right kernels.  Their span is compared with the full cage manifold, and the residual quotient

$$
\mathcal Q=\mathcal C_{(0,4)}/\mathcal K_{\rm regional}
$$

is represented canonically by the component of the cage manifold orthogonal to the regional-kernel span.


In [ ]:
from qlinks.caging import regional_cage_quotient

# Reuse the deterministic IPR construction described in the record-level analysis.
ipr_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=1.0e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64,
        ipr_candidate_count=32,
        ipr_random_seed=1234,
    ),
).run()
ipr_records = tuple(ipr_search[(0, 4)])

def embedded_cage(record):
    vector = np.zeros(ipr_search.hilbert_size, dtype=np.complex128)
    vector[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
    return vector

ipr_states = np.column_stack([embedded_cage(record) for record in ipr_records])
regional_supports = tuple(record.cage_state.support for record in ipr_records[:8])
quotient_report = regional_cage_quotient(
    square_build.kinetic,
    regional_supports,
    ipr_states,
    tolerance=1.0e-10,
)
quotient_overlaps = np.abs(ipr_states.conj().T @ quotient_report.quotient_basis) ** 2
pd.Series(quotient_report.to_summary_dict()), quotient_overlaps.ravel()


For the square-QDM data, the regional span is contained in the cage manifold to numerical precision, the intersection dimension is eight, and the quotient dimension is exactly one.  The squared overlaps of the canonical quotient vector with IPR records $0,\ldots,8$ are

$$
(0,0,0,0,0,0,0,0,1).
$$

Thus the quotient isolates the collective-cancellation record exactly, while its definition uses only subspaces and is independent of the chosen basis inside either subspace.  This quotient dimension is a discrete integer, but it is not automatically topological: it can change when a regional kernel appears/disappears or ceases to lie inside the full cage manifold.


## 7. Discrete invariants to test beyond the ordinary chiral index

The raw chiral index does not protect the cage-side zero modes, so the next candidates should be tested in the following order.

**1. Relative-kernel index.** Track the integer pair

$$
(\dim\mathcal K_{\rm regional},\;\dim\mathcal C/\mathcal K_{\rm regional}).
$$

For the present manifold it equals $(8,1)$.  This directly distinguishes eight independently closed cages from one collective mode.  Its stability must be tested under all allowed local deformations and under changes of the regional cover.

**2. $\mathbb Z_2$ signed holonomy.** Because the square-QDM amplitudes are real, each independent interference cycle carries a sign.  The sign can change only when an edge amplitude crosses zero or the graph pattern changes.  This is more genuinely discrete than the continuous holonomy magnitude and is the most immediate next candidate.

**3. Cycle-space parity or mod-2 homology.** Treat the active interference graph over $\mathbb Z_2$ and test whether the collective cage defines a nontrivial relative cycle class modulo the regional cycles.  A nonzero class would explain why it cannot be decomposed into the eight local kernels.

**4. Symmetry representation data.** Resolve the regional and quotient subspaces under translations, point-group operations, and winding-sector symmetries.  Discrete characters or symmetry eigenvalues can protect crossings or forbid mixing, although this would be symmetry-protected rather than intrinsic topology.

**5. Integer rank-jump invariant.** Track the interference nullity and quotient dimension over connected compatible parameter regions.  These are robust integers while the interference singular gap stays open, but they become topological only if different values cannot be connected without closing that gap.

The recommended next numerical step is the $\mathbb Z_2$ signed-holonomy and mod-2 relative-cycle test, because it is naturally tied to the already observed weighted-cycle flatness and can distinguish continuous fine tuning from a discrete obstruction.


## 8. Signed holonomy and relative mod-2 cycle space

We now test the two discrete cycle candidates on the same IPR-resolved manifold.  The full support-to-boundary graph is built from the union of the nine $(0,4)$ supports.  The eight compact supports define the regional cover.

For a real bipartite boundary matrix, the alternating sign around a cycle is invariant under independent row and column rescalings.  Separately, the graph cycle space can be formed over $\mathbb Z_2$ and quotiented by cycles generated inside the eight compact regions.

In [ ]:
from qlinks.caging import (
    diagnose_relative_mod2_cycles,
    diagnose_signed_boundary_holonomy,
    partition_cage_hamiltonian,
)

full_support = tuple(
    sorted(set().union(*(set(record.cage_state.support) for record in ipr_records)))
)
full_blocks = partition_cage_hamiltonian(square_build.kinetic, full_support)
full_column = {basis_index: column for column, basis_index in enumerate(full_support)}
regional_columns = tuple(
    tuple(full_column[basis_index] for basis_index in support)
    for support in regional_supports
)

signed_holonomy = diagnose_signed_boundary_holonomy(
    full_blocks.boundary,
    tolerance=1.0e-10,
)
relative_cycles = diagnose_relative_mod2_cycles(
    full_blocks.boundary,
    regional_columns,
    tolerance=1.0e-10,
)

{
    "signed_holonomy": signed_holonomy.to_summary_dict(),
    "relative_mod2_cycles": relative_cycles.to_summary_dict(),
}

The $\mathbb Z_2$ sign test is **not discriminating** for this square-QDM example: all 81 fundamental cycles have positive signed holonomy.  The eight compact regional graphs likewise contain five positive cycles each and no negative cycles.  This is expected for the unphased QDM kinetic matrix, whose active transition amplitudes share the same sign convention.

The relative cycle result is nontrivial:

$$
\dim Z_1^{\rm full}=81,
\qquad
\dim Z_1^{\rm regional}=40,
\qquad
\dim\left(Z_1^{\rm full}/Z_1^{\rm regional}\right)=41.
$$

Thus the full interference graph contains 41 independent mod-2 cycles that cannot be generated by cycles confined to the eight compact regions.  This supports the statement that the collective quotient state uses genuinely inter-regional connectivity.  However, the dimension 41 is a property of the graph pair, not yet a one-to-one invariant of the one-dimensional cage quotient.  The next task is to couple the cycle quotient to the **weighted cancellation equations** and identify which relative cycle combinations are actually selected by the collective cage vector.

## 9. Weighted cancellation matroid and the collective dependency class

The 41-dimensional relative mod-2 cycle space is too large because it knows only which edges exist.  The actual cage equation uses the weighted boundary matrix,

$$
B\psi=0.
$$

We therefore treat the columns of $B$ as a represented linear matroid.  A cage vector is a column dependency, while each compact four-configuration cage is a minimal regional dependency, or **circuit**.  The weighted relative dependency space is

$$
\ker B\Big/\operatorname{span}\!\left(\ker B_{\mathcal R_0},\ldots,\ker B_{\mathcal R_7}\right).
$$

Unlike graph homology, this quotient retains the QDM transition amplitudes.  It should therefore reduce the many possible inter-regional graph cycles to the combinations that actually satisfy destructive interference.

In [ ]:
from qlinks.caging import (
    combine_perturbations_from_coefficients,
    diagnose_boundary_cancellation_matroid,
    fixed_cage_manifold_compatibility_from_hamiltonians,
    fixed_cage_state_compatibility_from_hamiltonians,
    scan_boundary_cancellation_matroid,
    subspace_complement_basis,
)
from qlinks.caging.nullspace import nullspace_svd

weighted_matroid = diagnose_boundary_cancellation_matroid(
    full_blocks.boundary,
    regional_columns,
    tolerance=1.0e-10,
)
collective_local_state = ipr_states[np.asarray(full_support, dtype=np.int64), 8]
collective_quotient_overlap = abs(
    np.vdot(weighted_matroid.relative_dependency_basis[:, 0], collective_local_state)
) ** 2

{
    **weighted_matroid.to_summary_dict(),
    "collective_record_overlap": collective_quotient_overlap,
}

For the square QDM, the weighted dependency data are

$$
\dim\ker B=9,
\qquad
\dim\mathcal K_{\rm regional}=8,
\qquad
\dim\left(\ker B/\mathcal K_{\rm regional}\right)=1.
$$

All eight regional dependencies are matroid circuits.  The single weighted relative dependency has unit overlap with IPR record 8.  Thus the amplitude constraints reduce the 41 unweighted relative cycles to exactly the one collective cage class.

### A deformation that removes only the collective class

To test whether this integer quotient is physically meaningful, compare two local perturbation spaces:

1. perturbations preserving the complete nine-dimensional cage manifold;
2. perturbations preserving each of the eight compact cage vectors but not the full manifold.

The second space is the complement of the full-manifold compatibility space inside the common compatibility space of records $0$--$7$.  It gives a controlled deformation that leaves every regional circuit intact while lifting only the collective dependency.

In [ ]:
regional_fixed_reports = tuple(
    fixed_cage_state_compatibility_from_hamiltonians(
        square_build.hamiltonian,
        local_term_matrices,
        record.cage_state.support,
        record.cage_state.local_state,
        coefficient_field="real",
        tolerance=1.0e-10,
    )
    for record in ipr_records[:8]
)
regional_constraint_matrix = np.vstack(
    [report.constraint_matrix for report in regional_fixed_reports]
)
regional_exact_basis = nullspace_svd(
    regional_constraint_matrix,
    tolerance=1.0e-10,
)

manifold_exact_report = fixed_cage_manifold_compatibility_from_hamiltonians(
    square_build.hamiltonian,
    local_term_matrices,
    support=full_support,
    manifold_states=ipr_states,
    coefficient_field="real",
    tolerance=1.0e-10,
)
manifold_exact_basis = manifold_exact_report.compatible_coefficient_basis
regional_only_basis = subspace_complement_basis(
    regional_exact_basis,
    manifold_exact_basis,
    tolerance=1.0e-8,
)

{
    "regional_exact_dimension": regional_exact_basis.shape[1],
    "full_manifold_exact_dimension": manifold_exact_basis.shape[1],
    "regional_only_dimension": regional_only_basis.shape[1],
}

In [ ]:
def select_largest_boundary_direction(coefficient_basis):
    perturbations = combine_perturbations_from_coefficients(
        local_term_matrices,
        coefficient_basis,
    )
    boundary_norms = np.asarray(
        [
            np.linalg.norm(
                partition_cage_hamiltonian(perturbation, full_support)
                .boundary.toarray()
            )
            for perturbation in perturbations
        ]
    )
    return perturbations[int(np.argmax(boundary_norms))]

manifold_preserving_perturbation = select_largest_boundary_direction(
    manifold_exact_basis
)
collective_lifting_perturbation = select_largest_boundary_direction(
    regional_only_basis
)

base_boundary = full_blocks.boundary
manifold_boundary_perturbation = partition_cage_hamiltonian(
    manifold_preserving_perturbation,
    full_support,
).boundary
collective_lifting_boundary_perturbation = partition_cage_hamiltonian(
    collective_lifting_perturbation,
    full_support,
).boundary

matroid_parameters = (0.0, 1.0e-6, 1.0e-3, 1.0)
manifold_matroid_branch = scan_boundary_cancellation_matroid(
    base_boundary,
    manifold_boundary_perturbation,
    regional_columns,
    matroid_parameters,
    tolerance=1.0e-9,
)
collective_lifting_branch = scan_boundary_cancellation_matroid(
    base_boundary,
    collective_lifting_boundary_perturbation,
    regional_columns,
    matroid_parameters,
    tolerance=1.0e-9,
)

pd.DataFrame(
    [
        {"deformation": "full-manifold compatible", **point.to_summary_dict()}
        for point in manifold_matroid_branch.points
    ]
    + [
        {"deformation": "regional only", **point.to_summary_dict()}
        for point in collective_lifting_branch.points
    ]
)[
    [
        "deformation",
        "parameter",
        "dependency_dimension",
        "regional_dependency_span_dimension",
        "relative_dependency_dimension",
        "singular_gap",
    ]
]

The full-manifold-compatible branch keeps the discrete tuple

$$
(\dim\ker B,\dim\mathcal K_{\rm regional},\dim\mathcal Q)=(9,8,1)
$$

throughout the tested interval.  By contrast, a generic direction in the 13-dimensional regional-only space gives, for every nonzero sampled parameter,

$$
(9,8,1)\longrightarrow(8,8,0).
$$

The eight compact circuit cages remain exact, while the one collective dependency is lifted.  The newly opened singular value is linear at small deformation.  This is the cleanest discrete stability diagnostic obtained so far: the collective cage is exactly the relative weighted dependency class, and it can disappear without affecting any regional cage.

### Interpretation and limitation

The represented-matroid rank data are invariant under invertible row operations and nonzero rescaling of individual columns, matching the gauge freedom of the interference equations.  The integer relative nullity is locally constant until a relevant minor becomes nonzero or zero, equivalently until a singular value crosses zero.

This is stronger than ordinary graph homology and more directly tied to the cage mechanism.  It is nevertheless a finite-size, locality-relative invariant.  To promote it toward a topological classification, the next thermodynamic tests should determine whether the relative dependency dimension and its singular gap remain stable as the regional pattern is embedded or repeated at increasing system size.

## 10. Periodic repetition and the thermodynamic fate of the collective class

The weighted quotient dimension is a finite-size integer.  To ask whether it survives repeated embeddings, we form a one-dimensional block-circulant sequence of boundary maps,

$$
B_N=I_N\otimes B_0+\sum_d S_N^d\otimes C_d.
$$

A discrete Fourier transform reduces the problem exactly to the Bloch symbols

$$
B(k)=B_0+\sum_d e^{ikd}C_d,
\qquad k=\frac{2\pi j}{N}.
$$

This keeps the local boundary cell fixed and avoids constructing the exponentially large many-body Hilbert space.  We compare three possibilities for the one-dimensional collective quotient:

1. **independent repetition**, where the quotient is a flat zero band;
2. an **onsite quotient mass**, which preserves all eight regional circuits but lifts the collective class at every momentum;
3. a **balanced nearest-neighbor response**, proportional to $1-e^{ik}$, which leaves only the uniform $k=0$ collective mode and produces a gap closing as $1/N$.

The onsite and balanced responses below are constructed canonically from the weighted quotient vector.  They diagnose the algebraic thermodynamic possibilities; the balanced inter-cell response is not yet claimed to be generated by an allowed real-space-local square-QDM perturbation.


In [ ]:
from qlinks.caging import scan_periodic_boundary_cancellation_scaling

base_boundary_dense = (
    base_boundary.toarray()
    if hasattr(base_boundary, "toarray")
    else np.asarray(base_boundary)
)
collective_vector = weighted_matroid.relative_dependency_basis[:, 0]

# A rank-one response along the collective quotient.  Orthogonality to the
# regional dependency span makes it annihilate all eight regional circuits.
quotient_row = np.zeros(base_boundary_dense.shape[0], dtype=np.complex128)
quotient_row[0] = 1.0
collective_mass = np.outer(quotient_row, collective_vector.conj())
collective_mass *= (
    0.1 * np.linalg.norm(base_boundary_dense) / np.linalg.norm(collective_mass)
)

repeat_counts = (4, 8, 16, 32, 64, 128)
independent_scaling = scan_periodic_boundary_cancellation_scaling(
    base_boundary_dense,
    regional_columns,
    repeat_counts,
    tolerance=1.0e-9,
)
massive_scaling = scan_periodic_boundary_cancellation_scaling(
    base_boundary_dense,
    regional_columns,
    repeat_counts,
    coupling_terms=((0, collective_mass),),
    tolerance=1.0e-9,
)
derivative_scaling = scan_periodic_boundary_cancellation_scaling(
    base_boundary_dense,
    regional_columns,
    repeat_counts,
    coupling_terms=((0, collective_mass), (1, -collective_mass)),
    tolerance=1.0e-9,
)

scaling_table = pd.DataFrame(
    [
        {"family": family, **point.to_summary_dict()}
        for family, report in (
            ("independent", independent_scaling),
            ("onsite quotient mass", massive_scaling),
            ("balanced nearest neighbor", derivative_scaling),
        )
        for point in report.points
    ]
)
scaling_table[
    [
        "family",
        "n_repeats",
        "dependency_dimension",
        "regional_dependency_span_dimension",
        "relative_dependency_dimension",
        "relative_dependency_density",
        "relative_zero_momentum_count",
        "minimum_positive_relative_singular_gap",
    ]
]


In [ ]:
positive_gap_exponent = derivative_scaling.estimate_positive_relative_gap_exponent(
    minimum_repeats=16
)

plt.figure(figsize=(7, 4))
plt.loglog(
    derivative_scaling.repeat_counts,
    derivative_scaling.minimum_positive_relative_gaps,
    marker="o",
    label="smallest positive collective gap",
)
plt.xlabel("number of repeated boundary cells $N$")
plt.ylabel("relative singular gap")
plt.title(rf"Balanced collective response: fitted exponent {positive_gap_exponent:.3f}")
plt.legend()
plt.tight_layout()
plt.show()

{
    "independent_label": independent_scaling.scaling_label,
    "massive_label": massive_scaling.scaling_label,
    "balanced_label": derivative_scaling.scaling_label,
    "balanced_relative_dimensions": (
        derivative_scaling.relative_dependency_dimensions.tolist()
    ),
    "balanced_gap_exponent_N_ge_16": positive_gap_exponent,
}


The three sequences give sharply different thermodynamic conclusions:

- **Independent repetition:**
  $$
  \dim\mathcal Q_N=N,
  $$
  so the collective class forms an extensive flat zero band.  This is exact but not robust: it relies on the absence of quotient-coupling terms.

- **Onsite quotient mass:**
  $$
  \dim\mathcal Q_N=0
  $$
  for every $N$, while all $8N$ regional circuits remain.  The relative singular gap stays finite.  Hence the finite-cell integer $(8,1)$ alone does not protect the collective class under repeated local deformations.

- **Balanced nearest-neighbor response:**
  $$
  \dim\mathcal Q_N=1,
  \qquad
  \frac{\dim\mathcal Q_N}{N}\rightarrow0,
  $$
  and the smallest positive relative gap scales as
  $$
  \Delta_{\rm rel}(N)\propto N^{-1}.
  $$
  The surviving uniform mode is therefore a subextensive, gapless collective mode rather than a stable topological zero band.

This gives a practical thermodynamic criterion.  A candidate protection mechanism must forbid the onsite quotient mass and keep the relative singular gap from closing under every allowed finite-range $r$-local deformation.  The next physical test should derive the inter-cell boundary couplings from certified $4\times 8$, $4\times 12$, ... square-QDM padding embeddings instead of inserting an abstract quotient response.


## 11. Physical periodic embeddings from certified square-QDM padding

The Bloch benchmark above repeats a boundary matrix by direct sum.  A certified QDM padding instead produces a **tensor-product support** on the physical finite torus.  The two repetitions need not have the same nullity scaling.

Here we use the existing certified stripe-product unit cell and materialize only its finite product support for

$$
4\times4,\quad 8\times4,\quad 12\times4,\quad 16\times4.
$$

For each member we build every physical plaquette flip from the support to its one-hop exterior.  We then measure:

- the exact boundary nullity and interference singular gap;
- the overlap of the expected tensor-product cage with the boundary kernel;
- the space of independently varying plaquette kinetic couplings that preserve the cage;
- the corresponding uniform-on-support condition for plaquette potential couplings.

This is an exact finite-support calculation; it does not enumerate the full dimer Hilbert space.

In [ ]:
from qlinks.caging import (
    LocalQDMCageSearchConfig,
    RobustQDMLocalCageSearchConfig,
    SquareQDMPeriodicProductUnitCell,
    certify_square_qdm_periodic_product_sequence,
    robust_qdm_local_cage_search,
    scan_square_qdm_periodic_product_cancellation_scaling,
)

physical_local_config = LocalQDMCageSearchConfig(
    halo_layers=0,
    boundary_mode="relaxed",
    prune_inactive_local_basis_states=True,
    tolerance=1.0e-10,
    degenerate_basis_strategy="ipr",
    ipr_random_seed=1234,
)
physical_search_config = RobustQDMLocalCageSearchConfig(
    local_config=physical_local_config,
    region_strategies=("stripe",),
    stripe_widths=(1,),
    stripe_directions=(0, 1),
    max_regions_per_strategy=None,
    block_signatures=((0, 2),),
    max_records_per_region=2,
    min_blocks=2,
    max_blocks=None,
    max_product_support_size=2048,
    max_paddings_per_stage=100,
    max_paddings_per_packing=10,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)
physical_certified, physical_context = robust_qdm_local_cage_search(
    square_model,
    config=physical_search_config,
    return_context=True,
)

physical_candidates = []
for report_index, report in enumerate(physical_certified.reports):
    candidate = SquareQDMPeriodicProductUnitCell.from_padding(
        square_model,
        physical_context.blocks,
        report.padding,
        repeat_axis="x",
    )
    certificate = certify_square_qdm_periodic_product_sequence(candidate)
    if certificate.is_certified:
        physical_candidates.append((report_index, candidate, certificate))

physical_report_index, physical_unit_cell, physical_sequence = physical_candidates[0]
{
    "report_index": physical_report_index,
    "support_size_per_unit_cell": physical_unit_cell.support_size_per_unit_cell,
    "sequence_certified": physical_sequence.is_certified,
    "energy_density": physical_sequence.energy_density,
}

In [ ]:
physical_scaling = scan_square_qdm_periodic_product_cancellation_scaling(
    physical_unit_cell,
    repeat_counts=(1, 2, 3, 4),
    max_support_size=256,
    tolerance=1.0e-9,
)

physical_scaling_table = pd.DataFrame(
    [point.to_summary_dict() for point in physical_scaling.points]
)
physical_scaling_table[
    [
        "repeats",
        "system_size",
        "support_size",
        "shell_size",
        "boundary_nullity",
        "interference_gap",
        "product_state_kernel_weight",
        "kinetic_constraint_rank",
        "kinetic_compatible_fraction",
        "potential_constraint_rank",
    ]
]

In [ ]:
pair_rows = []
for point in physical_scaling.points:
    for left, right in point.kinetic_compatibility.equal_coupling_pairs:
        pair_rows.append(
            {
                "repeats": point.repeats,
                "system_size": point.system_size,
                "equal_kinetic_couplings": f"J[{left}] = J[{right}]",
            }
        )
pd.DataFrame(pair_rows)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    physical_scaling.repeat_counts,
    physical_scaling.interference_gaps,
    marker="o",
    label="interference gap",
)
plt.plot(
    physical_scaling.repeat_counts,
    physical_scaling.kinetic_constraint_ranks,
    marker="s",
    label="kinetic constraint rank",
)
plt.xlabel("number of repeated 4x4 cells")
plt.ylabel("diagnostic value")
plt.title("Physical tensor-product embedding")
plt.legend()
plt.tight_layout()
plt.show()

The physical tensor-product sequence behaves differently from the earlier direct-sum quotient benchmark:

$$
\dim\ker B_N=1
$$

for every explicitly materialized member, while

$$
\Delta_{\mathrm I}(N)=2
$$

within numerical precision.  The tensor-product cage is the unique boundary-kernel vector on these supports, and its kernel weight is one.

The local kinetic perturbation problem has

$$
\operatorname{rank}\mathcal O_K(N)=2N,
\qquad
\dim\ker\mathcal O_K(N)=14N,
$$

for the $16N$ independently varying plaquette kinetic coefficients.  Equivalently, two equality constraints are required in every repeated cell.  The explicit table above identifies the two equal-coupling pairs in each cell; the same local pairing pattern translates cell by cell.  The compatible kinetic fraction remains $7/8$, but the number of required tunings is extensive.

All independently varying plaquette potential terms are uniform on this product support:

$$
\operatorname{rank}\mathcal O_V(N)=0.
$$

Therefore the compact regional cage sequence has a finite interference gap **inside its compatible family**, but it is not stable against arbitrary $r$-local kinetic perturbations.  Its exact preservation requires an extensive set of local pairing conditions.  This is structural interference protection rather than intrinsic topological protection.

This construction embeds only the eight compact regional cages recovered by padding.  It does not provide a thermodynamic embedding of the ninth collective quotient state.  Finding such an embedding, or proving that no locality-preserving embedding exists, is now the central thermodynamic question for the collective record.

## 12. Non-factorized fixed-width extension of the collective cage

The compact stripe cages already admit a certified tensor-product sequence.  The remaining question is whether the one-dimensional collective quotient can extend along the fixed-width sequence without imposing product factorization.

We infer a finite-range column grammar directly from the 48 configurations supporting IPR record 8.  A range-$r$ grammar retains every cyclic window of $r$ consecutive columns observed at $4\times4$.  At larger length we enumerate **all** periodic dimer configurations accepted by this local language, restrict to the uniform potential shell, and solve the exact physical boundary equation on that support.  No amplitude factorization is imposed.

The resulting kernel is compared, basis independently, with the span of all translations of the certified stripe-product cage.  We define the integer locality-extension index

$$
\nu_{\rm ext}^{(r)}(L)
=
\dim\ker B_{\mathcal G_r(L)}
-
\dim\!\left(\ker B_{\mathcal G_r(L)}\cap\mathcal K_{\rm stripe}(L)\right).
$$

A positive value would certify a non-product collective extension within the chosen finite-range language.  A zero value means that the exact kernel is exhausted by translated stripe-product cages.

In [ ]:
from qlinks.caging import scan_square_qdm_collective_locality_extension

collective_record = ipr_records[8]
collective_support_configs = np.asarray(
    [square_build.basis.state(int(index)) for index in collective_record.support],
    dtype=np.int64,
)

collective_extension = scan_square_qdm_collective_locality_extension(
    square_model,
    collective_support_configs,
    physical_unit_cell,
    cases=((2, 8), (3, 8), (3, 12)),
    potential_per_column=1.0,
    max_words=10_000,
    max_product_support_size=128,
    dense_column_limit=512,
    maximum_nullity=16,
    ipr_restarts=64,
    tolerance=1.0e-9,
)

pd.DataFrame([point.to_summary_dict() for point in collective_extension.points])[
    [
        "window_size",
        "length",
        "support_size",
        "shell_size",
        "boundary_nullity",
        "interference_gap",
        "product_translation_span_dimension",
        "locality_extension_index",
        "minimum_principal_overlap",
        "localized_support_sizes",
    ]
]

For both the range-two and range-three languages at $8\times4$, and for the range-three language at $12\times4$, the exact result is

$$
\nu_{\rm ext}^{(r)}(L)=0.
$$

More explicitly,

$$
\dim\ker B_{\mathcal G_r(L)}=4,
$$

and all four principal overlaps with the translated stripe-product span equal one to numerical precision.  IPR localization gives four states with support sizes

$$
16\quad (8\times4),
\qquad
64\quad (12\times4),
$$

which are precisely the support sizes of the repeated local stripe products.  Their interference gaps remain finite: approximately $1.03$--$1.82$ at $8\times4$ depending on grammar range, and $1.86$ at $12\times4$.

Thus the local support patterns of the $4\times4$ collective state do extend as a configuration language, but their exact cancellation kernel does **not** carry an additional collective cage.  It collapses to the translated compact product family.  This is evidence for a finite-range locality obstruction to padding record 8, rather than evidence that the previous padding search merely missed a non-factorized state.

The conclusion is deliberately scoped: it rules out the collective extension inside the tested range-two and range-three column languages.  A larger-memory or genuinely tensor-network extension is not excluded.  The fixed-width sequence can be pushed further by this transfer-language approach; a true two-dimensional limit is deferred to the separate PEPS/TN program.

## 13. First discrete candidates after the extension test

The extension index $\nu_{\rm ext}^{(r)}(L)$ is already a basis-independent integer.  Here it acts as an **obstruction diagnostic** rather than a protecting topological invariant: the finite $4\times4$ relative class does not survive the tested fixed-width embeddings.

A natural next candidate is a real-sign cocycle.  For real amplitudes $\psi_w$ on cyclic column words, ask whether their signs can be generated by real local window weights,

$$
\operatorname{sgn}\psi_w
=
\eta_0\prod_e \eta_e^{N_{we}},
\qquad
\eta_0,\eta_e\in\{\pm1\}.
$$

The corresponding mod-two cokernel class is discrete and invariant under a global sign change.  We test it for all nine IPR records.

In [ ]:
from qlinks.caging import (
    diagnose_real_local_sign_obstruction,
    square_qdm_column_words,
)

sign_rows = []
for record_index, record in enumerate(ipr_records):
    record_configs = np.asarray(
        [square_build.basis.state(int(index)) for index in record.support],
        dtype=np.int64,
    )
    sign_report = diagnose_real_local_sign_obstruction(
        square_qdm_column_words(square_model, record_configs),
        record.local_state,
        window_size=3,
        tolerance=1.0e-10,
    )
    sign_rows.append({"record": record_index, **sign_report.to_summary_dict()})

pd.DataFrame(sign_rows)[
    [
        "record",
        "n_words",
        "n_local_windows",
        "obstruction_dimension",
        "magnitude_factorization_residual",
    ]
]

After quotienting the physically irrelevant global phase, the real-sign obstruction dimension is zero for **all nine** records.  Therefore this $\mathbb Z_2$ sign cocycle does not distinguish the collective state and should not be promoted as the desired invariant.

The current discrete conclusion is consequently:

1. the weighted relative dependency dimension $(8,1)$ distinguishes regional and collective cancellation at $4\times4$;
2. the fixed-width locality-extension index changes to zero at $8\times4$ and remains zero at $12\times4$ in the tested local languages;
3. simple signed holonomy, ordinary chiral index, integer-kernel torsion, and the real local-sign cocycle do not protect the collective class.

The next promising invariant should therefore involve a **finite-bond transfer representation**, not scalar local weights.  Concretely, one should classify the projective action of translation and reflection on the minimal boundary/bond space required to represent the cancellation amplitudes.  A quantized projective class or symmetry-fractionalization index could forbid the scalar quotient-mass deformation even when the ordinary graph, chiral, and sign invariants are trivial.  This is naturally compatible with the ongoing tensor-network development, while the fixed-width transfer problem can be studied first with exact finite-state matrices.

## Finite-bond transfer diagnostics

The fixed-width problem admits a second hierarchy beyond support grammar alone.  For a cyclic column amplitude, the ranks of all prefix--suffix flattenings give the exact finite-state bond complexity.  The maximum rank is the minimal exact open-boundary MPS bond dimension for the finite state, while a periodic MPS must have bond dimension at least the square root of this rank.

We also treat the exact boundary kernel as a finite transfer/bond space.  The commuting lattice translations resolve this space and the compact regional subspace into discrete momentum sectors.  Their multiplicity difference is a basis-independent symmetry label of the collective quotient.  This is a physical spatial representation, not automatically a virtual projective or SPT invariant.


In [ ]:
from qlinks.caging import (
    diagnose_cyclic_amplitude_bond_profile,
    diagnose_square_qdm_finite_bond_transfer_invariant,
)

bond_rows = []
for record_index, record in enumerate(ipr_records):
    record_configs = np.asarray(
        [square_build.basis.state(int(index)) for index in record.support],
        dtype=np.int64,
    )
    profile = diagnose_cyclic_amplitude_bond_profile(
        square_qdm_column_words(square_model, record_configs),
        record.local_state,
        tolerance=1.0e-10,
    )
    bond_rows.append({"record": record_index, **profile.to_summary_dict()})

pd.DataFrame(bond_rows)[
    [
        "record",
        "support_size",
        "cut_ranks",
        "exact_open_bond_dimension",
        "periodic_bond_dimension_lower_bound",
        "translation_eigenvalue",
        "translation_residual",
    ]
]


In [ ]:
product_bond_rows = []
for repeats in (1, 2, 3):
    instance = physical_unit_cell.instantiate(repeats)
    product_support = materialize_square_qdm_periodic_product_support(
        instance,
        max_support_size=128,
    )
    profile = diagnose_cyclic_amplitude_bond_profile(
        square_qdm_column_words(instance.model, product_support.configs),
        product_support.amplitudes,
        tolerance=1.0e-10,
    )
    product_bond_rows.append(
        {
            "repeats": repeats,
            "length": instance.model.lx,
            **profile.to_summary_dict(),
        }
    )

pd.DataFrame(product_bond_rows)[
    [
        "repeats",
        "length",
        "support_size",
        "cut_ranks",
        "exact_open_bond_dimension",
        "periodic_bond_dimension_lower_bound",
    ]
]


In [ ]:
full_support_configs = np.asarray(
    [square_build.basis.state(int(index)) for index in full_support],
    dtype=np.int64,
)
finite_bond_transfer = diagnose_square_qdm_finite_bond_transfer_invariant(
    square_model,
    full_support_configs,
    weighted_matroid.full_dependency_basis,
    weighted_matroid.regional_dependency_basis,
    tolerance=1.0e-9,
)

pd.Series(finite_bond_transfer.to_summary_dict()), pd.DataFrame(
    [sector.to_summary_dict() for sector in finite_bond_transfer.sectors]
)


The finite-bond calculation gives a sharp but ultimately negative projective-invariant result.

* The eight compact records have exact finite-cell open bond dimension at most four.  Their certified thermodynamic product sequence is even simpler: its exact bond dimension remains $D=2$ at lengths $4$, $8$, and $12$.
* The collective $4\times4$ record has cut ranks $(14,39,14)$, so its exact open-boundary bond dimension is $39$, and any periodic representation requires $D\geq7$.  The collective cancellation is therefore substantially more complex than the compact sequence.
* The one-dimensional quotient occupies only the fully trivial spatial sector: $k_x=k_y=0$, with even $x$ and $y$ reflections and quarter-turn character $+1$.  The eight-dimensional regional span contains no vector in this full trivial irrep, so the quotient is symmetry-distinct from every regional combination.
* Nevertheless, all translation, reflection, and dihedral group relations are represented linearly, with no nontrivial projective phase.  This symmetry label forbids rotation of the quotient into the regional span under symmetry-preserving deformations, but it does not prevent coupling to non-caged states in the same trivial sector or a symmetry-allowed change of the boundary rank.
* Most importantly, the range-two and range-three enlarged fixed-width kernels have zero non-product quotient.  Hence no persistent collective bond space exists on which a thermodynamic projective class could be defined within the tested local languages.

The natural next discrete object is therefore not a conventional one-dimensional SPT index.  For fixed width, the local boundary constraints can instead be assembled into a matrix over Laurent polynomials in the translation variable, $B(z)$.  Its kernel/cokernel module separates polynomially generated compact cage sequences from finite-size torsion classes.  Smith or determinantal data of this module can diagnose whether the $4\times4$ collective cage is a root-of-unity torsion mode rather than a freely extensible thermodynamic generator.


## Laurent-polynomial constraint-module invariant

For a fixed-width translation-invariant linear constraint problem, collect the finite-range boundary equations into a Laurent-polynomial symbol

$$
B(z)=\sum_{d=-R}^{R} z^d B_d,
$$

where multiplication by $z$ translates one unit cell.  The kernel over $\mathbb C(z)$ is the **free part** of the constraint module.  A free generator produces one zero mode at every Bloch momentum and hence an extensive periodic kernel.  Additional zero modes that occur only when $z$ is a root of unity form the **torsion part**.  Their primitive root orders are discrete determinantal data.

We first apply this diagnostic to the quotient-reduced algebraic families of Sec. 10.  The eight regional dependencies are removed from the domain, leaving only the sector containing the collective dependency.  We then test the observed physical fixed-width dimensions separately.  A necessary condition for any single fixed Laurent symbol is divisibility monotonicity: if $N\mid M$, every root of $z^N-1$ also belongs to $z^M-1$, so the torsion nullity at length $M$ cannot be smaller than at length $N$.


In [ ]:
from qlinks.caging import (
    diagnose_laurent_periodic_dimension_consistency,
    diagnose_laurent_polynomial_constraint_module,
)

quotient_domain_basis = subspace_complement_basis(
    np.eye(base_boundary_dense.shape[1], dtype=np.complex128),
    weighted_matroid.regional_dependency_basis,
    tolerance=1.0e-10,
)
base_quotient_symbol = base_boundary_dense @ quotient_domain_basis
mass_quotient_symbol = collective_mass @ quotient_domain_basis
laurent_repeat_counts = (1, 2, 3, 4, 6, 8)

laurent_reports = {
    "independent": diagnose_laurent_polynomial_constraint_module(
        ((0, base_quotient_symbol),),
        laurent_repeat_counts,
        tolerance=1.0e-9,
    ),
    "onsite quotient mass": diagnose_laurent_polynomial_constraint_module(
        ((0, base_quotient_symbol + mass_quotient_symbol),),
        laurent_repeat_counts,
        tolerance=1.0e-9,
    ),
    "balanced nearest neighbor": diagnose_laurent_polynomial_constraint_module(
        (
            (0, base_quotient_symbol + mass_quotient_symbol),
            (1, -mass_quotient_symbol),
        ),
        laurent_repeat_counts,
        tolerance=1.0e-9,
    ),
}

laurent_module_table = pd.DataFrame(
    [
        {
            "family": name,
            "module_label": report.module_label,
            "free_kernel_rank": report.free_kernel_rank,
            "torsion_orders": tuple(
                (entry.primitive_order, entry.multiplicity)
                for entry in report.torsion_orders
            ),
            "periodic_kernel_dimensions": tuple(
                point.total_kernel_dimension for point in report.periodic_points
            ),
        }
        for name, report in laurent_reports.items()
    ]
)

extension_dimension_by_length = {
    point.length: point.collective_quotient_dimension
    for point in collective_extension.points
}
physical_collective_consistency = diagnose_laurent_periodic_dimension_consistency(
    repeat_counts=(1, 2, 3),
    observed_dimensions=(
        weighted_matroid.relative_dependency_dimension,
        extension_dimension_by_length[8],
        extension_dimension_by_length[12],
    ),
    assumed_free_rank=0,
)

laurent_module_table, pd.Series(physical_collective_consistency.to_summary_dict())


The quotient-reduced algebraic families have the expected module structure:

* **Independent repetition:** one free Laurent generator, so $\dim\mathcal Q_N=N$.
* **Onsite quotient mass:** no free or torsion kernel generator.
* **Balanced nearest-neighbor response:** no free generator, but one primitive-order-one torsion class associated with the factor $1-z$.  It produces exactly one uniform $z=1$ mode for every periodic length.

The physical collective data behave differently.  In units of the $4\times4$ cell, the observed relative dimensions are

$$
q(1)=1,\qquad q(2)=0,\qquad q(3)=0.
$$

This violates the necessary Laurent-module divisibility law because $1$ divides both $2$ and $3$.  The mode at one cell necessarily lies at $z=1$, and $z=1$ is an allowed Bloch point for every periodic repetition.  Möbius inversion makes the contradiction explicit: the inferred primitive multiplicities are

$$
 m_1=1,\qquad m_2=-1,\qquad m_3=-1,
$$

which is impossible for a fixed module.

Therefore, within the tested range-two and range-three support languages, the $4\times4$ collective cage is **neither a free Laurent generator nor a root-of-unity torsion generator of any fixed finite-range translation-invariant constraint module**.  Its disappearance at $8\times4$ and $12\times4$ reflects a change of the admissible local constraint space itself, rather than the lifting of a conventional Laurent torsion branch.  This is a sharper fixed-width locality obstruction than the finite-size quotient dimension alone.

The conclusion is scoped to the chosen fixed-width local languages.  A larger auxiliary bond space can change the module presentation; this is precisely the possibility to revisit with the developing tensor-network construction before making a true two-dimensional claim.
